In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers==4.36.0 torch==2.1.0 scikit-learn==1.3.2 networkx==3.2.1
!pip install -q sinling sentencepiece accelerate
!pip install -q nltk rouge-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 3.3 MB/s eta 0:00:00
ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0)
ERROR: No matching distribution found for torch==2.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import os
import json
import torch
import numpy as np
from pathlib import Path
import nltk
from sinling import SinhalaTokenizer, POSTagger
from transformers import AutoModel, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('punkt', quiet=True)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(" All libraries imported successfully")

 All libraries imported successfully


In [ ]:
# Define paths
BASE_PATH = "/content/drive/MyDrive/SinBrief_Implementation"
TRAIN_PATH = f"{BASE_PATH}/Dataset/Train"
TEST_PATH = f"{BASE_PATH}/Dataset/Test"
OUTPUT_PATH = f"{BASE_PATH}/Outputs"
MODEL_PATH = f"{BASE_PATH}/Model"
MAIN_METHOD_PATH = f"{BASE_PATH}/Main_Method"

# Create directories if they don't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(MAIN_METHOD_PATH, exist_ok=True)

print("    Paths configured:")
print(f"   Train: {TRAIN_PATH}")
print(f"   Test: {TEST_PATH}")
print(f"   Output: {OUTPUT_PATH}")
print(f"   Model: {MODEL_PATH}")

    Paths configured:
   Train: /content/drive/MyDrive/SinBrief/Dataset/Train
   Test: /content/drive/MyDrive/SinBrief/Dataset/Test
   Output: /content/drive/MyDrive/SinBrief/Outputs
   Model: /content/drive/MyDrive/SinBrief/Model


In [ ]:
# Count train documents
train_files = [f for f in os.listdir(TRAIN_PATH) if f.endswith('.txt')]
print(f" Training documents found: {len(train_files)}")

# Count test documents
test_files = [f for f in os.listdir(TEST_PATH) if f.endswith('.txt')]
print(f" Test documents found: {len(test_files)}")

# Check a sample train document
if len(train_files) > 0:
    sample_path = os.path.join(TRAIN_PATH, train_files[0])
    with open(sample_path, 'r', encoding='utf-8') as f:
        sample_text = f.read()
    print(f"\n Sample document length: {len(sample_text)} characters")
    print(f" First 200 characters:\n{sample_text[:200]}...")
else:
    print(" WARNING: No training documents found!")

 Training documents found: 964
 Test documents found: 242

 Sample document length: 1081 characters
 First 200 characters:
ශ්‍රී ලංකා ප්‍රජාතාන්ත්‍රික සමාජවාදී ජනරජයේ
පාර්ලිමේන්තුව
2006 අංක 49 දරන එකතු කළ අගය මත බදු
(සංශෝධන) පනත

[සහතිකය සටහත් කළේ 2006 දෙසැම්බර් මස 15 වන දින]
ආණ්ඩුවේ නියමය පරිදි මුද්‍රණය කරන ලදී.
2006 දෙස...


In [ ]:
# Initialize Sinhala tools
print("Initializing Sinhala NLP tools...")
tokenizer_sin = SinhalaTokenizer()
pos_tagger = POSTagger()

# Test on sample text
test_sentence = "අධිකරණය විසින් තීන්දුව ප්‍රකාශ කරන ලදී"
tokens = tokenizer_sin.tokenize(test_sentence)
print(f"\n Sinhala Tokenizer working:")
print(f"   Input: {test_sentence}")
print(f"   Tokens: {tokens}")

try:
    pos_tags = pos_tagger.predict([tokens])[0]
    print(f" POS Tagger working:")
    print(f"   POS tags: {pos_tags}")
except Exception as e:
    print(f" POS Tagger test: {e}")
    print("   (This is okay, we'll handle it in the main code)")

Initializing Sinhala NLP tools...

 Sinhala Tokenizer working:
   Input: අධිකරණය විසින් තීන්දුව ප්‍රකාශ කරන ලදී
   Tokens: ['අධිකරණය', 'විසින්', 'තීන්දුව', 'ප්\u200dරකාශ', 'කරන', 'ලදී']
 POS Tagger working:
   POS tags: [('අධිකරණය', 'NNC'), ('විසින්', 'POST'), ('තීන්දුව', 'NNC'), ('ප්\u200dරකාශ', 'NCV'), ('කරන', 'VP'), ('ලදී', 'FS')]


In [ ]:
# Analyze document statistics
print("Analyzing dataset statistics...")

train_lengths = []
train_sentence_counts = []

for filename in train_files[:100]:  # Sample 100 docs for speed
    filepath = os.path.join(TRAIN_PATH, filename)
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            text = f.read()
            train_lengths.append(len(text))
            sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 10]
            train_sentence_counts.append(len(sentences))
    except:
        continue

# Calculate statistics
stats = {
    "total_train_docs": len(train_files),
    "total_test_docs": len(test_files),
    "avg_doc_length": int(np.mean(train_lengths)),
    "avg_sentences_per_doc": int(np.mean(train_sentence_counts)),
    "min_doc_length": int(np.min(train_lengths)),
    "max_doc_length": int(np.max(train_lengths))
}

print("\n Dataset Statistics:")
for key, value in stats.items():
    print(f"   {key}: {value}")

# Save statistics
with open(f"{OUTPUT_PATH}/dataset_statistics.json", 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f"\n Statistics saved to {OUTPUT_PATH}/dataset_statistics.json")

Analyzing dataset statistics...

 Dataset Statistics:
   total_train_docs: 964
   total_test_docs: 242
   avg_doc_length: 9902
   avg_sentences_per_doc: 35
   min_doc_length: 782
   max_doc_length: 57964

 Statistics saved to /content/drive/MyDrive/SinBrief/Outputs/dataset_statistics.json


In [ ]:
print("\n" + "="*60)
print(" DATA PREPARATION COMPLETE")
print("="*60)

print("\n    Setup verified:")
print(f"   ✓ {len(train_files)} training documents loaded")
print(f"   ✓ {len(test_files)} test documents loaded")
print(f"   ✓ Sinhala tokenizer working")
print(f"   ✓ Output directory created")

print("="*60)


 DATA PREPARATION COMPLETE

    Setup verified:
   ✓ 964 training documents loaded
   ✓ 242 test documents loaded
   ✓ Sinhala tokenizer working
   ✓ Output directory created
